The data in this project is a compilation of the monthly **output** of crude oil and condensate liquid from the major oil terminals/streams in Nigeria between January 2024 and June 2026.

The data was initially intended to be a compilation of crude oil production by field across Nigeria, however, gaining access to field-level data requires a paid license, but the terminal-level data has been made publicly available by the NUPRC.

This project is built to answer three questions:
1) Does each terminal show a downward, upward or stable trend in output, and how much does this vary from terminal to terminal?
2) To what degree does volatility exist in the output of each terminal on a month to month basis?
3) Do the top 20% of the terminals in the dataset account for 80% of the total output in compliance with the "Pareto principle"?

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_2024 = pd.read_csv("../data/raw/NUPRC_2024_production_raw.csv")
df_2025 = pd.read_csv("../data/raw/NUPRC_2025_production_raw.csv")
df_2026 = pd.read_csv("../data/raw/NUPRC_2026_production_raw.csv")

In [3]:
df_2024.head()

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
0,BONNY,Crude Oil,6351778.0,4604102.0,4260566.0,4158173.0,3605864.0,4536573.0,4584388.0,4894497.0,5589773.0,5765964.0,7051541.0,7274190.0
1,BONNY,Condensate,587495.0,583490.0,627057.0,543214.0,640026.0,568474.0,564610.0,584706.0,511021.0,494915.0,501329.0,509570.0
2,BONNY,Blend Total,6939273.0,5187592.0,4887623.0,4701387.0,4245890.0,5105047.0,5148998.0,5479203.0,6100794.0,6260879.0,7552870.0,7783759.0
3,BRASS,Crude Oil,735680.0,617189.0,686188.0,647053.0,635929.0,648849.0,664387.0,718104.0,842900.0,850352.0,775490.0,905544.0
4,BRASS,Condensate,160901.0,135498.0,158529.0,140724.0,153167.0,144677.0,148961.0,161633.0,201979.0,201451.0,179148.0,180417.0


In [4]:
df_2025.head()

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
0,BONNY,Crude Oil,7570173.0,5892185.0,7360738.0,7079233.0,6673050.0,6625912.0,7567542.0,5850190.0,5924385.0,7535315.0,7751499.0,7884015.0
1,BONNY,Condensate,574015.0,475882.0,290820.0,326253.0,423241.0,542024.0,507637.0,418278.0,393815.0,399509.0,419547.0,406781.0
2,BONNY,Blend Total,8144187.0,6368067.0,7651558.0,7405486.0,7096291.0,7167936.0,8075179.0,6268468.0,6318200.0,7934823.0,8171046.0,8290796.0
3,BRASS,Crude Oil,860312.0,726912.0,937437.0,619261.0,857711.0,710231.0,820821.0,846393.0,935010.0,946439.0,981254.0,914165.0
4,BRASS,Condensate,190629.0,149116.0,180191.0,127826.0,175028.0,167744.0,294502.0,309192.0,217677.0,220768.0,225872.0,265032.0


In [5]:
df_2026.head()

,Terminal/Stream,Liquid Type,January,February,March,April,May,June
0,BONNY,Crude Oil,232.30,262.88,258.00,278.68,277.04,301.93
1,BONNY,Condensate,12.57,13.18,13.78,16.43,16.83,16.34
2,BONNY,Blend Total,244.87,276.05,271.77,295.10,293.88,318.28
3,BRASS,Crude Oil,30.36,34.41,35.09,32.99,33.77,33.18
4,BRASS,Condensate,9.51,10.04,9.37,8.52,9.77,8.67


The values in the 2026 dataset are significantly lower than those of 2024 and 2024. This is because it is measured in thousand barells per day, unlike the data in 2024 and 2025 which is measured as the total number of barells produced. The unit conversion would be done in the cleaning phase.

In [6]:
print(df_2024.shape)
print(df_2025.shape)
print(df_2026.shape)

(56, 14)
(56, 14)
(54, 8)


In [7]:
df_2024.loc[:, "Terminal/Stream"] == df_2025.loc[:, "Terminal/Stream"]

0      True
1      True
2      True
3      True
4      True
5      True
6      True
7      True
8      True
9      True
10     True
11     True
12     True
13     True
14     True
15     True
16     True
17     True
18     True
19     True
20     True
21     True
22     True
23     True
24     True
25     True
26    False
27     True
28     True
29     True
30     True
31     True
32     True
33     True
34     True
35     True
36     True
37     True
38    False
39     True
40     True
41     True
42     True
43     True
44     True
45     True
46     True
47     True
48     True
49     True
50     True
51     True
52     True
53     True
54     True
55     True
Name: Terminal/Stream, dtype: bool

In [8]:
print(f"2024 Index 26 name: {df_2024.iloc[26, 0]}, 2025 Index 26 name: {df_2025.iloc[26, 0]}")
print(f"2024 Index 38 name: {df_2024.iloc[38, 0]}, 2025 Index 38 name: {df_2025.iloc[38, 0]}")

2024 Index 26 name: OTAKPIPO (Ex Ima Terminal), 2025 Index 26 name: OTAKPIPO
2024 Index 38 name: OYO, 2025 Index 38 name: OYO / OBODO


Comparing terminal names between 2024 and 2025: two mismatches found — 'OTAKPIPO (Ex Ima Terminal)' (2024) vs 'OTAKPIPO' (2025), and 'OYO' (2024) vs 'OYO / OBODO' (2025, likely reflecting a new grade addition). Same entity in both cases. Will standardize in cleaning, once I've checked whether 2026 introduces further variants.

In [9]:
#create a list of the Terminals/Streams in 2025 dataset
terminal_list_2025 = list(df_2025["Terminal/Stream"])
#create a list of the Terminals/Streams in 2026 dataset
terminal_list_2026 = list(df_2026["Terminal/Stream"])

#check which Terminals were removed from 2025 in 2026
removed_from_2025 = [terminal for terminal in terminal_list_2025 if terminal not in terminal_list_2026]
#check which Terminals were added in 2026 that were not in 2025
added_in_2026 = [terminal for terminal in terminal_list_2026 if terminal not in terminal_list_2025]
print(f"Terminals/Streams removed from 2025 in 2026: {removed_from_2025}")
print(f"Terminals/Streams added in 2026: {added_in_2026}")


Terminals/Streams removed from 2025 in 2026: ['ASARAMATORU (Ex Ima Terminal)', 'ANAMBRA BASIN', 'UKPOKITI']
Terminals/Streams added in 2026: ['CAWTHORNE']


As seen from the shape of the three datasets, data from the year 2026 has 2 fewer records(rows) than those of 2025 and 2024, as well as 6 fewer fields(columns).
The difference in fields is due to the fact that the 2026 entries end at June 2026.
THe difference in records, is caused by the removal of three Terminals from 2025 (ASARAMATORU (Ex Ima Terminal), ANAMBRA BASIN, UKPOKITI) and the addition of one in 2026 (CAWTHORNE).
Records which are not consistently included across all three datasets for 2024, 2025 and 2026 would be disregarded as this project seeks to analyze continuous production over the span of 2.5 years for each given Terminal.

In [10]:
df_2024.dtypes

Terminal/Stream        str
Liquid Type            str
January            float64
February           float64
March              float64
April              float64
May                float64
June               float64
July               float64
August             float64
September          float64
October            float64
November           float64
December           float64
dtype: object

In [11]:
df_2025.dtypes

Terminal/Stream        str
Liquid Type            str
January            float64
February           float64
March              float64
April              float64
May                float64
June               float64
July               float64
August             float64
September          float64
October            float64
November           float64
December           float64
dtype: object

In [12]:
df_2026.dtypes

Terminal/Stream        str
Liquid Type            str
January            float64
February           float64
March              float64
April              float64
May                float64
June               float64
dtype: object

Data types for each field are consistent with what is expected across the datasets.

In [13]:
null_2024_values = df_2024[df_2024.isnull().any(axis=1)]
null_2024_values

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
19,TULJA - OKWUIBOME,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,AJE,Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,AJE,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,AJE,Blend Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,ASARAMATORU (Ex Ima Terminal),Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,OKONO,Crude Oil,236705.0,NaN,NaN,NaN,195218.0,320167.0,278126.0,192153.0,347996.0,337559.0,363195.0,NaN
32,AJAPA,Crude Oil,28695.0,28351.0,6954.0,NaN,60199.0,74981.0,NaN,NaN,38099.0,70873.0,45219.0,81691.0
33,ANAMBRA BASIN,Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41,UKPOKITI,Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43,SEA EAGLE (EA),Crude Oil,395216.0,4265.0,NaN,374352.0,626868.0,632123.0,510925.0,621548.0,560093.0,613350.0,557724.0,529276.0


In [14]:
null_2025_values = df_2025[df_2025.isnull().any(axis=1)]
null_2025_values

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
19,TULJA - OKWUIBOME,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,AJE,Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,AJE,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,AJE,Blend Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,ASARAMATORU (Ex Ima Terminal),Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29,YOHO,Crude Oil,841110.0,735301.0,812328.0,806886.0,732578.0,783002.0,808663.0,803395.0,645913.0,NaN,NaN,NaN
32,AJAPA,Crude Oil,120796.0,40447.0,26327.0,28623.0,82996.0,37631.0,38156.0,NaN,77500.0,76666.0,45488.0,67198.0
33,ANAMBRA BASIN,Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41,UKPOKITI,Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,IMA,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
null_2026_values = df_2026[df_2026.isnull().any(axis=1)]
null_2026_values

,Terminal/Stream,Liquid Type,January,February,March,April,May,June
16,ODUDU (AMENAM BLEND),Condensate,NaN,NaN,NaN,0.24,0.24,0.33
19,TULJA - OKWUIBOME,Condensate,NaN,NaN,NaN,NaN,NaN,NaN
21,AJE,Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN
22,AJE,Condensate,NaN,NaN,NaN,NaN,NaN,NaN
23,AJE,Blend Total,NaN,NaN,NaN,NaN,NaN,NaN
29,YOHO,Crude Oil,0.12,NaN,NaN,NaN,NaN,NaN
36,OYO / OBODO,Crude Oil,0.80,0.77,0.78,0.00,NaN,NaN
47,IMA,Condensate,NaN,NaN,NaN,NaN,NaN,NaN


It can be seen that some records in each dataset are incomplete. Any records with a missing value would be dropped during analysis in cleaning. This is done to ensure ease and accuracy of analysis.

In [16]:
duplicates_2024 = df_2024[df_2024.duplicated(subset=["Terminal/Stream", "Liquid Type"], keep=False)]
duplicates_2024

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
19,TULJA - OKWUIBOME,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,TULJA - OKWUIBOME,Condensate,284045.0,264581.0,304064.0,299673.0,308497.0,280251.0,301720.0,283275.0,255131.0,296237.0,288743.0,282139.0


In [17]:
duplicates_2025 = df_2025[df_2025.duplicated(subset=["Terminal/Stream", "Liquid Type"], keep=False)]
duplicates_2025

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
19,TULJA - OKWUIBOME,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,TULJA - OKWUIBOME,Condensate,254417.0,272591.0,280584.0,267405.0,284381.0,276006.0,296915.0,303162.0,302246.0,320364.0,310201.0,318711.0


In [18]:
duplicates_2026 = df_2026[df_2026.duplicated(subset=["Terminal/Stream", "Liquid Type"], keep=False)]
duplicates_2026

,Terminal/Stream,Liquid Type,January,February,March,April,May,June
19,TULJA - OKWUIBOME,Condensate,NaN,NaN,NaN,NaN,NaN,NaN
49,TULJA - OKWUIBOME,Condensate,10.27,10.22,10.22,10.16,10.15,10.11


Seems to be a duplicate for the record of "TULJA - OKWUIBOME" Terminal in both 2025 and 2026 datasets. However, one of the entries is entirely null, so it would be discarded, and the entry with values would be used during analysis.

In [19]:
df_2024.describe()

,January,February,March,April,May,June,July,August,September,October,November,December
count,4.700000e+01,4.600000e+01,4.500000e+01,4.500000e+01,4.800000e+01,4.800000e+01,4.700000e+01,4.600000e+01,4.700000e+01,4.800000e+01,4.700000e+01,4.600000e+01
mean,3.886803e+06,3.464650e+06,3.558266e+06,3.459646e+06,3.364831e+06,3.333609e+06,3.587569e+06,3.767743e+06,3.511816e+06,3.513831e+06,3.873075e+06,4.018648e+06
std,9.598539e+06,8.439076e+06,8.487552e+06,8.387670e+06,8.388354e+06,8.297007e+06,8.857906e+06,9.219967e+06,8.656225e+06,8.843177e+06,9.612682e+06,9.940376e+06
min,6.413000e+03,4.265000e+03,6.954000e+03,6.427000e+03,3.896000e+03,6.970000e+03,6.789000e+03,5.246000e+03,5.821000e+03,6.910000e+02,5.462000e+03,5.498000e+03
25%,3.136330e+05,2.756445e+05,3.218740e+05,2.996730e+05,3.091998e+05,3.145505e+05,3.139880e+05,3.040902e+05,2.701315e+05,3.159208e+05,2.880235e+05,2.924245e+05
50%,9.453010e+05,1.036540e+06,1.061497e+06,8.892320e+05,8.322980e+05,8.149315e+05,9.281170e+05,9.379035e+05,1.044879e+06,1.108908e+06,9.546380e+05,1.185612e+06
75%,3.344956e+06,3.410196e+06,3.006372e+06,3.306128e+06,2.884685e+06,2.891586e+06,3.108878e+06,2.977762e+06,2.795426e+06,2.991664e+06,3.316910e+06,3.468504e+06
max,5.095380e+07,4.464866e+07,4.458200e+07,4.342305e+07,4.552568e+07,4.500597e+07,4.754465e+07,4.869112e+07,4.632869e+07,4.768198e+07,5.071454e+07,5.169436e+07


In [20]:
df_2025.describe()

,January,February,March,April,May,June,July,August,September,October,November,December
count,4.700000e+01,4.700000e+01,4.700000e+01,4.700000e+01,4.700000e+01,4.700000e+01,4.700000e+01,4.600000e+01,4.700000e+01,4.600000e+01,4.700000e+01,4.700000e+01
mean,4.113849e+06,3.565249e+06,3.767954e+06,3.868917e+06,3.912937e+06,3.893134e+06,4.057606e+06,3.939309e+06,3.614519e+06,3.867868e+06,3.703512e+06,3.728139e+06
std,1.023145e+07,8.844670e+06,9.364226e+06,9.607048e+06,9.703991e+06,9.684700e+06,1.006551e+07,9.664645e+06,8.971540e+06,9.501998e+06,9.225472e+06,9.337117e+06
min,5.934000e+03,4.508000e+03,5.221000e+03,4.865000e+03,5.043000e+03,4.438000e+03,4.741000e+03,4.741000e+03,4.590000e+03,4.666000e+03,4.565000e+03,4.616000e+03
25%,3.747460e+05,3.035360e+05,2.834890e+05,3.009920e+05,3.188700e+05,2.782285e+05,3.056510e+05,3.266940e+05,2.950640e+05,2.970438e+05,2.631515e+05,2.369545e+05
50%,1.050941e+06,1.080581e+06,1.145283e+06,1.051648e+06,1.178584e+06,1.161601e+06,1.189629e+06,1.251420e+06,1.152687e+06,1.294378e+06,1.212574e+06,1.119722e+06
75%,3.173460e+06,3.011052e+06,3.012108e+06,3.067994e+06,3.152703e+06,2.886904e+06,3.100470e+06,3.232327e+06,2.791990e+06,2.627290e+06,2.492530e+06,2.118758e+06
max,5.386188e+07,4.681470e+07,4.971706e+07,5.049921e+07,5.138048e+07,5.091136e+07,5.308074e+07,5.058000e+07,4.743544e+07,4.951318e+07,4.797161e+07,4.787469e+07


In [21]:
df_2026.describe()

,January,February,March,April,May,June
count,48.000000,47.000000,47.000000,48.000000,47.000000,47.000000
mean,121.997708,115.479362,118.110213,124.798750,130.962128,133.871064
std,308.584827,283.550812,294.975139,315.283581,326.805024,333.549098
min,0.120000,0.770000,0.780000,0.000000,0.090000,0.330000
25%,8.587500,8.435000,8.380000,7.795000,7.220000,7.355000
50%,40.450000,36.320000,41.570000,40.555000,41.720000,40.680000
75%,82.162500,67.165000,81.285000,76.657500,86.135000,88.860000
max,1627.460000,1483.950000,1546.090000,1663.410000,1700.800000,1735.400000


Significantly lower values in the description of the 2026 dataset is expected, as it was recorded in a different unit (thousand barrels per month). Conversion would be done in the cleaning phase.

In [31]:
df_2024.columns == df_2025.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True])

In [34]:
df_2025.columns[0:8] == df_2026.columns

array([ True,  True,  True,  True,  True,  True,  True,  True])